# 🏠 Análisis del Mercado de Airbnb en Nueva York
## Fase 6: Storytelling, Análisis Final y Conclusiones

---

**Proyecto:** Business Intelligence — Equipo 10  
**Dataset:** NYC Airbnb 2019 (Inside Airbnb)  
**Framework:** CRISP-DM  

---

### Pregunta de negocio

> ¿Qué factores geográficos, de tipo de propiedad y de perfil del anfitrión influyen en el precio y la demanda de los alojamientos de Airbnb en Nueva York?

Este notebook sintetiza los hallazgos de todas las fases anteriores del proyecto y presenta los resultados de los indicadores clave de desempeño (KPIs) junto con conclusiones, recomendaciones y limitaciones del análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

CORAL = '#E07B54'
STEEL = '#4A90A4'
PALETTE_DISTRITOS = ['#E07B54', '#4A90A4', '#6BB88F', '#B57BA6', '#F0C27F']

df = pd.read_csv('../data/processed/airbnb_clean.csv')

print(f'Registros cargados: {len(df):,}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

---

## 1. Resumen del proceso CRISP-DM

El proyecto siguió el framework CRISP-DM en 6 fases iterativas:

| Fase | Nombre | Descripción | Entregable |
|------|--------|-------------|------------|
| 1 | Comprensión del problema | Exploración inicial del dataset, definición del problema de negocio y preguntas clave | `01_exploracion_inicial.ipynb` |
| 2 | Formulación de hipótesis | Definición de 3 hipótesis estadísticas y aplicación de tests (Kruskal-Wallis, Mann-Whitney U) | `02_hipotesis_planteamiento.ipynb` |
| 3 | Preparación de datos | Limpieza, imputación de nulos, eliminación de outliers, ingeniería de variables (`host_type`) | `03_preparacion.ipynb` |
| 4 | Definición de KPIs | Diseño de 5 indicadores clave alineados con las hipótesis y el problema de negocio | `docs/fase4_kpis.md` |
| 5 | Dashboard | Visualización interactiva de KPIs para toma de decisiones | Dashboard BI |
| 6 | Análisis y conclusiones | Síntesis de hallazgos, storytelling, recomendaciones y limitaciones | **Este notebook** |

**Dataset final:** 48,410 registros × 9 columnas — precio filtrado entre \$10 y \$799 (percentil 99), sin valores nulos.

---

## 2. Resultados de hipótesis estadísticas

Las tres hipótesis planteadas en la Fase 2 fueron evaluadas con un nivel de significancia α = 0.05. Los resultados fueron:

| # | Hipótesis | Test aplicado | Estadístico | p-valor | Decisión |
|---|-----------|--------------|-------------|---------|----------|
| H1 | El precio promedio es igual en los 5 distritos | Kruskal-Wallis | H = 6,942.9 | ≈ 0.0000 | **Se rechaza H₀** |
| H2 | El precio promedio es igual entre tipos de alojamiento | Kruskal-Wallis | H = 22,408.1 | ≈ 0.0000 | **Se rechaza H₀** |
| H3 | No hay diferencia en reseñas entre tipos de anfitrión | Mann-Whitney U | U = 234,786,039 | 1.44e-93 | **Se rechaza H₀** |

**Las tres hipótesis nulas fueron rechazadas con evidencia estadística extremadamente sólida**, lo que confirma que el distrito, el tipo de alojamiento y el perfil del anfitrión son factores determinantes en el comportamiento del mercado.

> *Nota metodológica:* Se utilizaron pruebas no paramétricas porque los datos de precio y reseñas no siguen una distribución normal (Shapiro-Wilk p ≈ 0.0000 en todos los grupos) y presentan heterocedasticidad (Levene p ≈ 0.0000).

---

## 3. Indicadores Clave de Desempeño (KPIs)

A continuación se calculan y visualizan los 5 KPIs definidos en la Fase 4.

In [ ]:
# ─────────────────────────────────────────────
# KPI 1: Precio promedio por distrito
# ─────────────────────────────────────────────
orden_distritos = ['Manhattan', 'Brooklyn', 'Queens', 'Staten Island', 'Bronx']

kpi1 = (
    df.groupby('neighbourhood_group')['price']
    .mean()
    .reindex(orden_distritos)
    .reset_index()
)
kpi1.columns = ['Distrito', 'Precio Promedio (USD)']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    kpi1['Distrito'],
    kpi1['Precio Promedio (USD)'],
    color=PALETTE_DISTRITOS,
    edgecolor='white',
    height=0.6
)
for bar, val in zip(bars, kpi1['Precio Promedio (USD)']):
    ax.text(val + 2, bar.get_y() + bar.get_height() / 2,
            f'${val:.2f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Precio promedio por noche (USD)')
ax.set_title('KPI 1 — Precio promedio por distrito', fontsize=14, fontweight='bold', pad=12)
ax.set_xlim(0, 210)
ax.invert_yaxis()
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

print(kpi1.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# KPI 2: Precio promedio por tipo de alojamiento
# ─────────────────────────────────────────────
orden_tipos = ['Entire home/apt', 'Private room', 'Shared room']

kpi2 = (
    df.groupby('room_type')['price']
    .mean()
    .reindex(orden_tipos)
    .reset_index()
)
kpi2.columns = ['Tipo de alojamiento', 'Precio Promedio (USD)']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    kpi2['Tipo de alojamiento'],
    kpi2['Precio Promedio (USD)'],
    color=[CORAL, STEEL, '#6BB88F'],
    edgecolor='white',
    width=0.5
)
for bar, val in zip(bars, kpi2['Precio Promedio (USD)']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 2,
            f'${val:.2f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Precio promedio por noche (USD)')
ax.set_title('KPI 2 — Precio promedio por tipo de alojamiento', fontsize=14, fontweight='bold', pad=12)
ax.set_ylim(0, 230)
sns.despine()
plt.tight_layout()
plt.show()

print(kpi2.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# KPI 3: Promedio de reseñas por tipo de anfitrión
# ─────────────────────────────────────────────
kpi3 = (
    df.groupby('host_type')['number_of_reviews']
    .mean()
    .reset_index()
)
kpi3.columns = ['Tipo de anfitrión', 'Reseñas promedio']

delta = kpi3.loc[kpi3['Tipo de anfitrión'] == 'Profesional', 'Reseñas promedio'].values[0] - \
        kpi3.loc[kpi3['Tipo de anfitrión'] == 'Ocasional', 'Reseñas promedio'].values[0]

fig, ax = plt.subplots(figsize=(7, 5))
colors = [STEEL if t == 'Profesional' else CORAL for t in kpi3['Tipo de anfitrión']]
bars = ax.bar(
    kpi3['Tipo de anfitrión'],
    kpi3['Reseñas promedio'],
    color=colors,
    edgecolor='white',
    width=0.45
)
for bar, val in zip(bars, kpi3['Reseñas promedio']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3,
            f'{val:.2f}', ha='center', fontsize=13, fontweight='bold')

ax.annotate(
    f'Δ = +{delta:.2f} reseñas\n(anfitriones profesionales)',
    xy=(1, kpi3.loc[kpi3['Tipo de anfitrión'] == 'Profesional', 'Reseñas promedio'].values[0]),
    xytext=(1.35, kpi3['Reseñas promedio'].mean()),
    fontsize=10,
    color=STEEL,
    arrowprops=dict(arrowstyle='->', color=STEEL)
)

ax.set_ylabel('Número promedio de reseñas')
ax.set_title('KPI 3 — Promedio de reseñas por tipo de anfitrión', fontsize=14, fontweight='bold', pad=12)
ax.set_ylim(0, 38)
sns.despine()
plt.tight_layout()
plt.show()

# También comparar precios por tipo de anfitrión
precio_por_host = df.groupby('host_type')['price'].mean().reset_index()
precio_por_host.columns = ['Tipo de anfitrión', 'Precio promedio (USD)']
print('Reseñas promedio:')
print(kpi3.to_string(index=False))
print('\nPrecio promedio:')
print(precio_por_host.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# KPI 4: Nivel de actividad del alojamiento
# Heatmap: reviews_per_month por distrito y tipo de alojamiento
# ─────────────────────────────────────────────
kpi4 = (
    df.groupby(['neighbourhood_group', 'room_type'])['reviews_per_month']
    .mean()
    .unstack('room_type')
    .reindex(orden_distritos)
    .reindex(columns=orden_tipos)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    kpi4,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Reseñas por mes (promedio)'}
)
ax.set_title('KPI 4 — Nivel de actividad: reseñas mensuales por distrito y tipo de alojamiento',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Tipo de alojamiento')
ax.set_ylabel('Distrito')
plt.tight_layout()
plt.show()

print('\nActividad promedio general por distrito:')
print(df.groupby('neighbourhood_group')['reviews_per_month'].mean()
      .reindex(orden_distritos)
      .rename('reviews_per_month (promedio)')
      .to_string())

In [ ]:
# ─────────────────────────────────────────────
# KPI 6: Ingreso potencial estimado
# Fórmula: price × (365 - availability_365)
# ─────────────────────────────────────────────
df['ingreso_potencial'] = df['price'] * (365 - df['availability_365'])

kpi6_distrito = (
    df.groupby('neighbourhood_group')['ingreso_potencial']
    .mean()
    .reindex(orden_distritos)
    .reset_index()
)
kpi6_distrito.columns = ['Distrito', 'Ingreso Potencial Promedio (USD)']

# Top 10 barrios por ingreso potencial
top10_barrios = (
    df.groupby('neighbourhood')['ingreso_potencial']
    .mean()
    .nlargest(10)
    .reset_index()
)
top10_barrios.columns = ['Barrio', 'Ingreso Potencial Promedio (USD)']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico izquierdo: por distrito
bars = axes[0].barh(
    kpi6_distrito['Distrito'],
    kpi6_distrito['Ingreso Potencial Promedio (USD)'],
    color=PALETTE_DISTRITOS,
    edgecolor='white',
    height=0.6
)
for bar, val in zip(bars, kpi6_distrito['Ingreso Potencial Promedio (USD)']):
    axes[0].text(val + 200, bar.get_y() + bar.get_height() / 2,
                 f'${val:,.0f}', va='center', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Ingreso potencial promedio anual (USD)')
axes[0].set_title('KPI 6 — Ingreso potencial por distrito', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=axes[0], left=True, bottom=True)

# Gráfico derecho: top 10 barrios
bars2 = axes[1].barh(
    top10_barrios['Barrio'],
    top10_barrios['Ingreso Potencial Promedio (USD)'],
    color=STEEL,
    edgecolor='white',
    height=0.6
)
for bar, val in zip(bars2, top10_barrios['Ingreso Potencial Promedio (USD)']):
    axes[1].text(val + 500, bar.get_y() + bar.get_height() / 2,
                 f'${val:,.0f}', va='center', fontsize=9)
axes[1].set_xlabel('Ingreso potencial promedio anual (USD)')
axes[1].set_title('KPI 6 — Top 10 barrios por ingreso potencial', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=axes[1], left=True, bottom=True)

plt.tight_layout()
plt.show()

print('Ingreso potencial promedio por distrito:')
print(kpi6_distrito.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# Dashboard resumen — todos los KPIs en una figura
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Dashboard — Análisis del Mercado Airbnb NYC 2019',
             fontsize=16, fontweight='bold', y=1.01)

# ── KPI 1: precio por distrito
ax1 = fig.add_subplot(2, 3, 1)
ax1.barh(kpi1['Distrito'], kpi1['Precio Promedio (USD)'],
         color=PALETTE_DISTRITOS, edgecolor='white', height=0.6)
for i, (_, row) in enumerate(kpi1.iterrows()):
    ax1.text(row['Precio Promedio (USD)'] + 1, i, f"${row['Precio Promedio (USD)']:.0f}",
             va='center', fontsize=9)
ax1.set_title('KPI 1 · Precio por distrito', fontweight='bold')
ax1.set_xlabel('USD / noche')
ax1.invert_yaxis()
sns.despine(ax=ax1, left=True, bottom=True)

# ── KPI 2: precio por tipo de alojamiento
ax2 = fig.add_subplot(2, 3, 2)
ax2.bar(kpi2['Tipo de alojamiento'], kpi2['Precio Promedio (USD)'],
        color=[CORAL, STEEL, '#6BB88F'], edgecolor='white', width=0.5)
for i, (_, row) in enumerate(kpi2.iterrows()):
    ax2.text(i, row['Precio Promedio (USD)'] + 1, f"${row['Precio Promedio (USD)']:.0f}",
             ha='center', fontsize=9)
ax2.set_title('KPI 2 · Precio por tipo', fontweight='bold')
ax2.set_ylabel('USD / noche')
ax2.tick_params(axis='x', labelsize=8)
sns.despine(ax=ax2)

# ── KPI 3: reseñas por tipo de anfitrión
ax3 = fig.add_subplot(2, 3, 3)
colors3 = [STEEL if t == 'Profesional' else CORAL for t in kpi3['Tipo de anfitrión']]
ax3.bar(kpi3['Tipo de anfitrión'], kpi3['Reseñas promedio'],
        color=colors3, edgecolor='white', width=0.45)
for i, (_, row) in enumerate(kpi3.iterrows()):
    ax3.text(i, row['Reseñas promedio'] + 0.2, f"{row['Reseñas promedio']:.1f}",
             ha='center', fontsize=9)
ax3.set_title('KPI 3 · Reseñas por anfitrión', fontweight='bold')
ax3.set_ylabel('Reseñas promedio')
sns.despine(ax=ax3)

# ── KPI 4: heatmap de actividad
ax4 = fig.add_subplot(2, 3, 4)
sns.heatmap(kpi4, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax4,
            cbar_kws={'shrink': 0.8})
ax4.set_title('KPI 4 · Actividad mensual', fontweight='bold')
ax4.set_xlabel('Tipo')
ax4.set_ylabel('Distrito')
ax4.tick_params(axis='x', labelsize=7, rotation=15)
ax4.tick_params(axis='y', rotation=0)

# ── KPI 6: ingreso potencial por distrito
ax5 = fig.add_subplot(2, 3, 5)
ax5.barh(kpi6_distrito['Distrito'], kpi6_distrito['Ingreso Potencial Promedio (USD)'],
         color=PALETTE_DISTRITOS, edgecolor='white', height=0.6)
for i, (_, row) in enumerate(kpi6_distrito.iterrows()):
    ax5.text(row['Ingreso Potencial Promedio (USD)'] + 100, i,
             f"${row['Ingreso Potencial Promedio (USD)']:,.0f}", va='center', fontsize=8)
ax5.set_title('KPI 6 · Ingreso potencial', fontweight='bold')
ax5.set_xlabel('USD / año (estimado)')
ax5.invert_yaxis()
ax5.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
sns.despine(ax=ax5, left=True, bottom=True)

# ── Top 10 barrios (KPI 6 complementario)
ax6 = fig.add_subplot(2, 3, 6)
ax6.barh(top10_barrios['Barrio'], top10_barrios['Ingreso Potencial Promedio (USD)'],
         color=STEEL, edgecolor='white', height=0.6)
ax6.set_title('KPI 6 · Top 10 barrios', fontweight='bold')
ax6.set_xlabel('USD / año (estimado)')
ax6.invert_yaxis()
ax6.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax6.tick_params(axis='y', labelsize=8)
sns.despine(ax=ax6, left=True, bottom=True)

plt.tight_layout()
plt.savefig('../dashboard_resumen.png', bbox_inches='tight', dpi=120)
plt.show()
print('Dashboard guardado como dashboard_resumen.png')

---

## 4. Hallazgos clave

A partir del análisis estadístico y los KPIs calculados, se identificaron los siguientes hallazgos principales:

### 📍 Hallazgo 1 — La ubicación es el factor de mayor impacto en el precio

Manhattan tiene el precio promedio más alto del mercado (\$172.90/noche), casi el **doble** del Bronx (\$83.86/noche). Brooklyn ocupa un punto intermedio (\$115.92/noche) y se consolida como la alternativa más competitiva para viajeros que buscan equilibrio entre precio y accesibilidad. Queens y Staten Island ofrecen precios similares (~\$94/noche), pero con mayor disponibilidad anual, lo que sugiere menor demanda relativa.

### 🏠 Hallazgo 2 — El tipo de alojamiento explica una brecha de precio de casi 3x

Una propiedad completa cuesta en promedio \$189.10/noche, frente a \$83.39 de una habitación privada y \$64.20 de una habitación compartida. Esto representa una diferencia del **194%** entre los extremos. Dado que el 51.7% del mercado son propiedades completas, la composición de la oferta influye directamente en el precio promedio de cada distrito.

### 👤 Hallazgo 3 — Los anfitriones profesionales acumulan más demanda, pero cobran menos

Los anfitriones con múltiples propiedades (profesionales) reciben en promedio **28.98 reseñas**, frente a **20.55** de los ocasionales, una diferencia de +8.4 reseñas (p = 1.44×10⁻⁹³). Sin embargo, sus precios son más bajos (\$127.93 vs \$142.54). Esto sugiere una estrategia de **volumen sobre precio**: mayor ocupación a tarifas más competitivas.

### 📈 Hallazgo 4 — Brooklyn y Manhattan lideran en actividad mensual

Los alojamientos en Brooklyn y Manhattan muestran la mayor rotación mensual, especialmente en habitaciones privadas. Queens presenta baja actividad relativa a pesar de sus precios moderados, lo que puede indicar menor visibilidad en la plataforma o menor atractivo turístico para los perfiles de demanda más frecuentes.

### 💰 Hallazgo 5 — Manhattan domina el ingreso potencial, pero hay oportunidades en Brooklyn

El ingreso potencial estimado (precio × noches ocupadas estimadas) es máximo en Manhattan, pero varios barrios de Brooklyn como **Williamsburg, Bedford-Stuyvesant y Bushwick** aparecen en el top 10, lo que refleja una combinación favorable de precios razonables con alta tasa de ocupación. Para anfitriones con propiedades fuera de Manhattan, Brooklyn representa la mejor oportunidad de maximizar ingresos.

---

## 5. Recomendaciones

### Para anfitriones

| Perfil | Recomendación | Fundamento |
|--------|--------------|------------|
| **Nuevo anfitrión en Manhattan** | Fijar precio inicial por debajo del promedio del barrio (\$172.90) para generar primeras reseñas rápidamente | KPI 3: los anfitriones con más reseñas tienen mayor ocupación |
| **Anfitrión ocasional con habitación privada** | Considerar precio entre \$75–\$90 para ser competitivo con el promedio del tipo (\$83.39) | KPI 2: habitaciones privadas tienen alta demanda relativa |
| **Anfitrión en Queens o Staten Island** | Reducir disponibilidad mínima de noches para aumentar la tasa de conversión; los precios ya son competitivos | KPI 4: baja actividad mensual en estos distritos |
| **Anfitrión con múltiples propiedades** | Priorizar ocupación sobre precio: el modelo profesional de \$127.93 + alta rotación supera al modelo ocasional en ingreso total estimado | KPI 6 + KPI 3 |

### Para la plataforma Airbnb

1. **Activar programas de visibilidad** en Queens y Staten Island: la baja actividad mensual en estos distritos no se debe al precio sino a menor exposición. Campañas de marketing específicas podrían equilibrar la demanda en el mercado.

2. **Incentivar la profesionalización responsable**: los anfitriones profesionales generan más reseñas y mayor satisfacción. Sin embargo, la concentración de propiedades en manos de pocos operadores puede afectar la diversidad de la oferta. Un modelo de incentivos escalonado favorecería el equilibrio.

3. **Desarrollar herramientas de pricing dinámico**: la diferencia de precio entre tipos de alojamiento y distritos es predecible y consistente. Ofrecer recomendaciones de precio en tiempo real basadas en estacionalidad y competencia local ayudaría a optimizar ingresos de los anfitriones y la conversión de la plataforma.

---

## 6. Limitaciones del análisis

Este análisis fue desarrollado con rigor metodológico, pero presenta limitaciones importantes que deben considerarse al interpretar los resultados:

1. **Dataset histórico (2019):** Los datos corresponden a un período pre-pandemia. El mercado de Airbnb en Nueva York cambió significativamente a partir de 2020 por restricciones regulatorias y el impacto del COVID-19. Los valores calculados no reflejan el estado actual del mercado.

2. **`availability_365` como proxy de ocupación:** Esta variable indica los días en que el listado *está disponible*, no los días efectivamente ocupados. Un alojamiento con alta disponibilidad podría tener baja demanda real o estar recién publicado. El ingreso potencial estimado (KPI 6) asume 100% de conversión de noches disponibles, por lo que representa un techo máximo, no una predicción realista.

3. **Ausencia de datos de costos:** No se cuenta con información sobre gastos operativos del anfitrión (limpieza, mantenimiento, comisiones de plataforma, impuestos). Por lo tanto, el ingreso potencial es bruto, no una estimación de ganancia neta.

4. **Sesgo de selección:** El dataset incluye únicamente listados activos en la plataforma en el momento de la extracción. Listados inactivos, suspendidos o que dejaron de operar no están representados, lo que puede sesgar las distribuciones de precio hacia arriba.

5. **Clasificación de anfitriones:** La variable `host_type` fue derivada de `calculated_host_listings_count`: un listado = Ocasional, más de un listado = Profesional. Esta clasificación es una simplificación. Un anfitrión con dos propiedades y uno con veinte se tratan de la misma manera, lo que puede subestimar las diferencias reales entre segmentos.

6. **Análisis transversal:** El dataset es un corte en el tiempo, no una serie temporal. No es posible analizar tendencias de precio o demanda a lo largo del año ni identificar estacionalidad.

---

## 7. Conclusiones finales

El análisis del mercado de Airbnb en Nueva York confirma con alta significancia estadística que **el precio y la demanda de los alojamientos están determinados por tres factores clave: el distrito geográfico, el tipo de alojamiento y el perfil del anfitrión**.

Manhattan concentra los precios más altos y el mayor ingreso potencial estimado, pero la brecha con Brooklyn se está cerrando gracias a una combinación de precios competitivos y alta actividad mensual. El tipo de alojamiento introduce la mayor variabilidad de precio, con propiedades completas cobrando casi tres veces más que las habitaciones compartidas. Y el perfil del anfitrión revela una dinámica interesante: los anfitriones profesionales han adoptado una estrategia de volumen que les permite acumular más reseñas a tarifas más bajas, mientras que los ocasionales mantienen precios más altos pero con menor rotación.

Estos hallazgos responden directamente la pregunta de negocio planteada en la Fase 1 y ofrecen una base sólida para decisiones estratégicas tanto de anfitriones individuales como de la plataforma. Un anfitrión que comprenda estas dinámicas puede posicionar su propiedad de manera más efectiva; la plataforma, a su vez, puede usar estos patrones para optimizar su algoritmo de recomendaciones y sus herramientas de soporte a anfitriones.

---

*Proyecto finalizado — Framework CRISP-DM completo (Fases 1–6)*  
*Dataset: NYC Airbnb 2019 | Registros analizados: 48,410 | Equipo 10*